In [2]:
"""
denoise_background_subtract.py

Step 2 of the glass pipeline: remove static artifacts (dust on lens,
bubbles, uneven background) by subtracting a pixel-wise TIME-MEDIAN
background from every frame.

Why median, not mean: a static artifact is identical in every frame,
so both the mean and median remove it equally well. But real particles
move, so at any given pixel they're only "in the way" some of the time.
The median only cares about the majority state at each pixel, so as
long as a background pixel is uncovered more than 50% of the time,
subtracting it doesn't damage real particle signal. The mean would
partially subtract particles too, especially in crowded regions --
which is what we want to avoid.

MEMORY NOTE: an earlier version of this script loaded ALL frames into
memory to compute the median, which crashed at 1500x1500 px x 4000
frames (~9GB raw, likely 20-30GB+ peak during np.median). Fixed here
by computing the median from a random SUBSAMPLE of frames instead of
all of them. This works because static artifacts are identical in
every frame -- a few hundred sample frames pin down the median just
as well as all 4000 would. The subtraction pass itself is fully
streamed (one frame in memory at a time), same as crop_and_trim.py.

Input:  the cropped/trimmed .avi from crop_and_trim.py
Output: a same-size, same-length .avi, uint8, where each frame is
        (frame - median_background), offset by +128 so "no change"
        sits at gray (128) and both brighter/darker-than-background
        directions are preserved.
"""

import cv2
import numpy as np
import os

# ----------------------------------------------------------------------
# 1. CONFIG -- edit these values
# ----------------------------------------------------------------------

INPUT_PATH = "/Volumes/Expansion/recordings/theo_seth_1_cropped_0-4000.avi"  # <-- fill in: path to the cropped .avi from step 1

OUTPUT_DIR = os.path.dirname(INPUT_PATH)  # <-- fill in: folder to save the denoised .avi into
OUTPUT_FILENAME = "denoised.avi"

OFFSET = 128  # so "no change from background" = mid-gray

N_SAMPLE_FRAMES = 300  # how many frames to sample to compute the background
RANDOM_SEED = 0        # fixed seed so the sample (and result) is reproducible


def main():
    if INPUT_PATH is None or OUTPUT_DIR is None:
        raise ValueError("Set INPUT_PATH and OUTPUT_DIR before running.")

    output_path = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

    # ------------------------------------------------------------------
    # 2. PASS 1: read total frame count, pick a random subsample of
    #    frame indices, load ONLY those into memory, compute the
    #    pixel-wise median across the subsample. We deliberately do NOT
    #    load all frames -- static artifacts are identical in every
    #    frame, so a few hundred sampled frames pin down the median just
    #    as well as all 4000 would, at a fraction of the memory.
    # ------------------------------------------------------------------

    cap = cv2.VideoCapture(INPUT_PATH)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {INPUT_PATH}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 1.0
    print(f"Input video: {total_frames} frames, {width}x{height} px")

    n_sample = min(N_SAMPLE_FRAMES, total_frames)
    rng = np.random.default_rng(RANDOM_SEED)
    sample_indices = np.sort(rng.choice(total_frames, size=n_sample, replace=False))
    print(f"Sampling {n_sample} of {total_frames} frames to estimate background...")

    sample_frames = np.empty((n_sample, height, width), dtype=np.uint8)
    for k, idx in enumerate(sample_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            raise IOError(f"Could not read sampled frame {idx}")
        if frame.ndim == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        sample_frames[k] = frame

    cap.release()
    print("Computing pixel-wise median background from sample...")

    # median across the sample axis (axis=0) -> one (height, width) image
    background = np.median(sample_frames, axis=0)
    del sample_frames  # free that memory before pass 2

    # ------------------------------------------------------------------
    # 3. PASS 2: stream through the FULL video one frame at a time,
    #    subtract the background, offset, clip, write out. We never
    #    hold more than one frame (plus the single background image)
    #    in memory at once here -- same streaming approach as
    #    crop_and_trim.py.
    # ------------------------------------------------------------------

    cap = cv2.VideoCapture(INPUT_PATH)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {INPUT_PATH}")

    fourcc = cv2.VideoWriter_fourcc(*"FFV1")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height), isColor=False)
    if not writer.isOpened():
        raise IOError(f"Could not open VideoWriter for: {output_path}")

    background_f32 = background.astype(np.float32)
    first_frame_preview = None
    last_frame_preview = None
    n_written = 0

    for i in range(total_frames):
        ret, frame = cap.read()
        if not ret:
            print(f"WARNING: failed to read frame {i} on subtraction pass; stopping early.")
            break
        if frame.ndim == 3:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # do the subtraction in float so we don't wrap around at 0 like
        # uint8 would (e.g. 5 - 10 should be -5, not 251)
        diff = frame.astype(np.float32) - background_f32
        shifted = diff + OFFSET
        clipped = np.clip(shifted, 0, 255).astype(np.uint8)

        writer.write(clipped)
        n_written += 1

        if i == 0:
            first_frame_preview = clipped.copy()
        last_frame_preview = clipped.copy()

        if i % 500 == 0:
            print(f"  processed frame {i}")

    cap.release()
    writer.release()
    print(f"\nDone. Wrote {n_written} denoised frames to:\n  {output_path}")

    # ------------------------------------------------------------------
    # 4. SANITY CHECK -- save the background image itself, plus
    #    first/last denoised frames, so you can visually confirm
    #    the artifacts (dust/bubble/background) are actually gone.
    # ------------------------------------------------------------------

    preview_dir = os.path.join(OUTPUT_DIR, "denoise_previews")
    os.makedirs(preview_dir, exist_ok=True)
    cv2.imwrite(os.path.join(preview_dir, "background_median.png"),
                background.astype(np.uint8))
    cv2.imwrite(os.path.join(preview_dir, "first_frame_denoised.png"),
                first_frame_preview)
    cv2.imwrite(os.path.join(preview_dir, "last_frame_denoised.png"),
                last_frame_preview)
    print(f"Saved preview images (background + first/last denoised frame) to:\n  {preview_dir}")


if __name__ == "__main__":
    main()

Input video: 4000 frames, 1572x1432 px
Sampling 300 of 4000 frames to estimate background...
Computing pixel-wise median background from sample...
  processed frame 0
  processed frame 500
  processed frame 1000
  processed frame 1500
  processed frame 2000
  processed frame 2500
  processed frame 3000
  processed frame 3500

Done. Wrote 4000 denoised frames to:
  /Volumes/Expansion/recordings/denoised.avi
Saved preview images (background + first/last denoised frame) to:
  /Volumes/Expansion/recordings/denoise_previews
